In [15]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
import time

# Function to extract MFCCs from audio files
def extract_mfccs(file_path):
    try:
        y, sr = librosa.load(file_path, duration=1)  # Load 1-second audio
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=100)
        return np.mean(mfccs.T, axis=0)  # Average MFCCs over time
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Data Loading and Feature Extraction
male_folder = "/home/feliciano/male_lobsters"  # Replace with your actual male folder name if different
female_folder = "/home/feliciano/female_lobsters" # Replace with your actual female folder name if different

data = []
labels = []

for filename in os.listdir(male_folder):
    file_path = os.path.join(male_folder, filename)
    if os.path.isfile(file_path) and filename.endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("male")

for filename in os.listdir(female_folder):
    file_path = os.path.join(female_folder, filename)
    if os.path.isfile(file_path) and filename.endswith(('.wav', '.mp3', '.ogg', '.flac')):
        mfccs = extract_mfccs(file_path)
        if mfccs is not None:
            data.append(mfccs)
            labels.append("female")
            print("mfcc length: ", len(mfccs))

if not data:
    print("No audio files found in the specified folders.")
    exit()

X = np.array(data)
y = np.array(labels)

# Label Encoding and Data Scaling
le = LabelEncoder()
y = le.fit_transform(y)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1D-Deep CNN Model with 5 Layers
model_name = "1D-Deep CNN, # Layers = 5"
model = Sequential()
model.add(Conv1D(64, 3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(2))
model.add(Conv1D(128, 3, activation='relu'))
model.add(MaxPooling1D(2))
model.add(Conv1D(256, 3, activation='relu'))
model.add(MaxPooling1D(2))
model.add(Conv1D(512, 3, activation='relu'))
model.add(MaxPooling1D(2))
model.add(Conv1D(1024, 3, activation='relu'))
model.add(MaxPooling1D(2))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # Binary classification
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Reshape data for CNN input
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Model Training and Evaluation
start_time = time.time()
model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, verbose=0)
end_time = time.time()
inference_time = (end_time - start_time) * 1000 #milliseconds

y_pred_prob = model.predict(X_test_cnn)
y_pred = (y_pred_prob > 0.5).astype(int)

# Calculate Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_pred_prob)

# Print Results
print(f"{model_name} Performance:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  AUC-ROC: {auc_roc:.4f}")
print(f"  Inference Time (ms): {inference_time:.4f}")
print("\nNote: MAP-50 is not directly applicable for binary classification.")

mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc length:  100
mfcc lengt

/home/feliciano/anaconda3/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_70 (Conv1D)              │ (None, 98, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_67 (MaxPooling1D) │ (None, 49, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_71 (Conv1D)              │ (None, 47, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_68 (MaxPooling1D) │ (None, 23, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_72 (Conv1D)              │ (None, 21, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_69 (MaxPooling1D) │ (None, 10, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_73 (Conv1D)              │ (None, 8, 512)         │       393,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_70 (MaxPooling1D) │ (None, 4, 512)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_74 (Conv1D)              │ (None, 2, 1024)        │     1,573,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_71 (MaxPooling1D) │ (None, 1, 1024)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_11 (Flatten)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,222,465 (8.48 MB)

 Trainable params: 2,222,465 (8.48 MB)

 Non-trainable params: 0 (0.00 B)

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1D-Deep CNN, # Layers = 5 Performance:
  Accuracy: 0.9309
  Precision: 0.9310
  Recall: 0.9572
  F1-Score: 0.9439
  AUC-ROC: 0.9836
  Inference Time (ms): 47562.1748

Note: MAP-50 is not directly applicable for binary classification.
